In [1]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

api_key = os.getenv("GOOGLE_API_KEY")

print(api_key is not None)

True


In [3]:
%pip install python-dotenv pandas langchain langchain-community langchain-google-genai chromadb

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using

In [2]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

api_key = os.getenv("GOOGLE_API_KEY")

print(api_key is not None)

True


In [6]:
df = pd.read_csv("../data/cleaned_data.csv")

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,rag_text
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",Title: Dick Johnson Is Dead\nType: Movie\nDire...
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",Title: Blood & Water\nType: TV Show\nDirector:...
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,Title: Ganglands\nType: TV Show\nDirector: Jul...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",Title: Jailbirds New Orleans\nType: TV Show\nD...
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,Title: Kota Factory\nType: TV Show\nDirector: ...


In [7]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'rag_text'],
      dtype='str')

In [8]:
df.shape

(8807, 13)

In [9]:
sweden_df = df[df["country"].str.contains("Sweden", case=False, na=False)]

other_df = df[
    ~df["country"].str.contains("Sweden", case=False, na=False)
].sample(n=300, random_state=42)

df_rag = pd.concat([sweden_df, other_df]).drop_duplicates().reset_index(drop=True)

df_rag.shape

(342, 13)

In [10]:
df_rag["country"].value_counts().head(10)

country
United States     99
Unknown           37
India             28
Sweden            13
United Kingdom    12
South Korea        8
Japan              7
Spain              7
Turkey             6
Indonesia          4
Name: count, dtype: int64

In [11]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=row["rag_text"],
        metadata={
            "title": row["title"],
            "type": row["type"],
            "country": row["country"],
            "release_year": int(row["release_year"])
        }
    )
    for _, row in df_rag.iterrows()
]

len(documents)

342

In [12]:
documents[0]

Document(metadata={'title': 'Young Royals', 'type': 'TV Show', 'country': 'Sweden', 'release_year': 2021}, page_content='Title: Young Royals\nType: TV Show\nDirector: Unknown\nCast: Edvin Ryding, Omar Rudberg, Malte Gårdinger, Frida Argento, Nikita Uggla, Pernilla August\nCountry: Sweden\nRelease year: 2021\nRating: TV-MA\nDuration: 1 Season\nCategories: International TV Shows, Romantic TV Shows, TV Dramas\nDescription: Prince Wilhelm adjusts to life at his prestigious new boarding school, Hillerska, but following his heart proves more challenging than anticipated.')

In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x0000019E843A68D0>, model='models/gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [3]:
df = pd.read_csv("../data/cleaned_data.csv")

sweden_df = df[df["country"].str.contains("Sweden", case=False, na=False)]

other_df = df[
    ~df["country"].str.contains("Sweden", case=False, na=False)
].sample(n=30, random_state=42)

df_rag = pd.concat([sweden_df, other_df]).drop_duplicates().reset_index(drop=True)

df_rag.shape

(72, 13)

In [4]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=row["rag_text"],
        metadata={
            "title": row["title"],
            "type": row["type"],
            "country": row["country"],
            "release_year": int(row["release_year"])
        }
    )
    for _, row in df_rag.iterrows()
]

len(documents)

72

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x000001D3B287CA50>, model='models/gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [6]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="../vectorstore"
)

vectorstore

C:\Users\paulh\AppData\Local\Temp\ipykernel_19072\2729680518.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [7]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D3B622DDD0>, search_kwargs={'k': 5})

In [8]:
test_docs = retriever.invoke("Which titles are TV Shows from Sweden?")

len(test_docs)

5

In [9]:
for doc in test_docs:
    print(doc.metadata)
    print(doc.page_content[:500])
    print("-" * 80)

{'title': 'Bonus Family', 'country': 'Sweden', 'type': 'TV Show', 'release_year': 2019}
Title: Bonus Family
Type: TV Show
Director: Unknown
Cast: Vera Vitali, Erik Johansson, Fredrik Hallgren, Petra Mede, Frank Dorsin, Jacob Lundqvist, Amanda Lindh, Marianne Mörck, Barbro Svensson, Ann Petrén, Johan Ulveson, Leo Razzak, Felix Engström
Country: Sweden
Release year: 2019
Rating: TV-MA
Duration: 3 Seasons
Categories: International TV Shows, TV Dramas
Description: A new couple, their exes and their children navigate the emotional challenges and tricky logistics of blended family life i
--------------------------------------------------------------------------------
{'release_year': 2017, 'type': 'TV Show', 'title': 'The Most Beautiful Hands of Delhi', 'country': 'Sweden'}
Title: The Most Beautiful Hands of Delhi
Type: TV Show
Director: Unknown
Cast: Björn Kjellman, Joy Sengupta, Natasha Jayetileke
Country: Sweden
Release year: 2017
Rating: TV-MA
Duration: 1 Season
Categories: International

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

template = """
You are a helpful assistant that answers questions based only on the context below.

Use only the information in the context.
If the answer is not found in the context, say:
"I cannot find that information in the dataset."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D3B622DDD0>, search_kwargs={'k': 5})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant that answers questions based only on the context below.\n\nUse only the information in the context.\nIf the answer is not found in the context, say:\n"I cannot find that information in the dataset."\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n'), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': 

In [11]:
answer = rag_chain.invoke("Which titles are TV Shows from Sweden?")

print(answer)

The following titles are TV Shows from Sweden:

*   Bonus Family
*   The Most Beautiful Hands of Delhi
*   Caliphate
*   Quicksand
*   Young Royals


In [12]:
answer = rag_chain.invoke("Give me three Netflix titles from Sweden and explain what they are about.")

print(answer)

Here are three Netflix titles from Sweden and what they are about:

*   **The Most Beautiful Hands of Delhi**: A stalled career, failed marriage and receding hairline drive a middle-aged Swedish man to try to revitalize his life by visiting India.
*   **Bonus Family**: A new couple, their exes and their children navigate the emotional challenges and tricky logistics of blended family life in this Swedish dramedy.
*   **Quicksand**: After a tragedy at a school sends shock waves through a wealthy Stockholm suburb, a seemingly well-adjusted teen finds herself on trial for murder.


In [13]:
answer = rag_chain.invoke("Which Netflix titles in the dataset are from Iceland?")

print(answer)

And Breathe Normally


In [14]:
answer = rag_chain.invoke("Which Netflix titles in the dataset are from Mars?")

print(answer)

I cannot find that information in the dataset.


## Testresultat för RAG-applikationen

I den här notebooken byggde vi en liten RAG-applikation med det rensade Netflix-datasetet.

Arbetsflödet var:

1. Läsa in den rensade CSV-filen.
2. Skapa ett mindre RAG-urval eftersom embedding-API:t hade begränsningar.
3. Omvandla raderna till LangChain Document-objekt.
4. Skapa embeddings med Gemini.
5. Spara embeddings i ChromaDB.
6. Skapa en retriever.
7. Koppla retrievern till Gemini med en RAG-kedja.
8. Testa frågor där svaret finns och frågor där svaret inte finns.

På grund av API-begränsningar använde vi ett mindre urval av datasetet:
- alla titlar kopplade till Sverige
- 30 slumpmässiga andra titlar

Hela datasetet rensades och sparades i preprocessing-notebooken, men själva RAG-demonstrationen använder ett mindre urval.